<a href="https://colab.research.google.com/github/kaz3107/IS4487/blob/main/assignment_11_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IS 4487 Assignment 11: Predicting Airbnb Prices with Regression

In this assignment, you will:
- Load the Airbnb dataset you cleaned and transformed in Assignment 7
- Build a linear regression model to predict listing price
- Interpret which features most affect price
- Try to improve your model using only the most impactful predictors
- Practice explaining your findings to a business audience like a host, pricing strategist, or city partner

## Why This Matters

Pricing is one of the most important levers for hosts and Airbnb’s business teams. Understanding what drives price — and being able to predict it accurately — helps improve search results, revenue management, and guest satisfaction.

This assignment gives you hands-on practice turning a cleaned dataset into a predictive model. You’ll focus not just on code, but on what the results mean and how you’d communicate them to stakeholders.

<a href="https://colab.research.google.com/github/Stan-Pugsley/is_4487_base/blob/main/Assignments/assignment_11_regression.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>



## Original Source: Dataset Description

The dataset you'll be using is a **detailed Airbnb listing file**, available from [Inside Airbnb](https://insideairbnb.com/get-the-data/).

Each row represents one property listing. The columns include:

- **Host attributes** (e.g., host ID, host name, host response time)
- **Listing details** (e.g., price, room type, minimum nights, availability)
- **Location data** (e.g., neighborhood, latitude/longitude)
- **Property characteristics** (e.g., number of bedrooms, amenities, accommodates)
- **Calendar/booking variables** (e.g., last review date, number of reviews)

The schema is consistent across cities, so you can expect similar columns regardless of the location you choose.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score


## 1. Load Your Transformed Airbnb Dataset

**Business framing:**  
Before building any models, we must start with clean, prepared data. In Assignment 7, you exported a cleaned version of your Airbnb dataset. You’ll now import that file for analysis.

### Do the following:
- Import your CSV file called `cleaned_airbnb_data_7.csv`.   (Note: If you had significant errors with assignment 7, you can use the file named "airbnb_listings.csv" in the DataSets folder on GitHub as a backup starting point.)
- Use `pandas` to load and preview the dataset


In [2]:
# Load cleaned Airbnb data
df = pd.read_csv('/content/cleaned_airbnb_data_7.csv')

# Preview the data
display(df.head())

# Check rows and columns
print("Shape:", df.shape)

,id,listing_url,scrape_id,last_scraped,source,name,description,picture_url,host_id,host_url,...,log_price,price_scaled,minimum_nights_scaled,review_group,price_per_guest,long_term_stay,room_Entire home/apt,room_Hotel room,room_Private room,room_Shared room
0,1188641052152277018,https://www.airbnb.com/rooms/1188641052152277018,20260615212226,2026-06-16,city scrape,Spacious unit at heart of SEA,"Located in the heart of Seattle, This is a stu...",https://a0.muscache.com/pictures/miso/Hosting-...,32542619,https://www.airbnb.com/users/show/32542619,...,4.814783,0.004329,1.102957,Low,40.773333,0,True,False,False,False
1,849077848951991421,https://www.airbnb.com/rooms/849077848951991421,20260615212226,2026-06-16,city scrape,"Blueground | Central District, gym & bbq, nr d...",Feel at home wherever you choose to live with ...,https://a0.muscache.com/pictures/prohost-api/H...,107434423,https://www.airbnb.com/users/show/107434423,...,NaN,NaN,1.102957,Low,NaN,0,True,False,False,False
2,599205596133828323,https://www.airbnb.com/rooms/599205596133828323,20260615212226,2026-06-16,city scrape,purple room,simple room facing front yard in centrally lo...,https://a0.muscache.com/pictures/miso/Hosting-...,48441443,https://www.airbnb.com/users/show/48441443,...,4.238445,0.002243,1.102957,Low,34.150000,0,False,False,True,False
3,846763,https://www.airbnb.com/rooms/846763,20260615212226,2026-06-16,city scrape,Right at Home in Ravenna,"Our home is a lovely peaceful place, and we wi...",https://a0.muscache.com/pictures/62990267/b8fe...,4421734,https://www.airbnb.com/users/show/4421734,...,4.213312,0.002177,1.102957,High,66.580000,0,False,False,True,False
4,1512026480622905433,https://www.airbnb.com/rooms/1512026480622905433,20260615212226,2026-06-16,city scrape,"3 Min Walk to Capitol Hill Station, Brand New ...",Brand new 3BR/3BA townhome in the heart of Cap...,https://a0.muscache.com/pictures/hosting/Hosti...,44486963,https://www.airbnb.com/users/show/44486963,...,6.293419,0.020451,-0.435566,Medium,67.500000,0,True,False,False,False


Shape: (7769, 97)


### Reflection:
1. What business information does the dataset include?
2. How many rows and columns are present?

### ✍️ Your Response: 🔧
1. The dataset includes information about Airbnb listings such as price, location, room type, host information, minimum nights, reviews, and property characteristics.

2. The dataset has 7,769 rows and 97 columns.

## 2. Drop Columns Not Useful for Modeling

**Business framing:**  
Some columns — like post IDs or text — may not help us predict price and could add noise or bias.

### Do the following:
- Drop columns like `post_id`, `title`, `descr`, `details`, and `address` if they’re still in your dataset

In [3]:
# Drop columns not useful for modeling
drop_cols = ['id', 'post_id', 'title', 'descr', 'details', 'address']

df = df.drop(columns=drop_cols, errors='ignore')

# Check the result
print("Shape after dropping columns:", df.shape)

Shape after dropping columns: (7769, 96)


### Reflection:
1. What columns did you drop, and why?
2. What risks might occur if you included them in your model?

### ✍️ Your Response: 🔧
1. I dropped the id column because it is only an identifier and is not useful for predicting Airbnb prices. The other suggested text columns were not present in my cleaned dataset.

2. If I included these types of columns, they could add unnecessary information or noise to the model and make it harder to interpret.

## 3. Explore Relationships Between Numeric Features

**Business framing:**  
Understanding how features relate to each other — and to the target — helps guide feature selection and modeling.

### Do the following:
- Generate a correlation matrix
- Identify which variables are strongly related to `price`

In [4]:
# Create correlation matrix for numeric columns
corr_matrix = df.corr(numeric_only=True)

# Show correlations with price
price_corr = corr_matrix['price'].sort_values(ascending=False)

print(price_corr)

price                          1.000000
price_quote_price_per_night    1.000000
price_scaled                   1.000000
price_per_guest                0.693336
log_price                      0.605968
                                 ...   
host_total_listings_count           NaN
host_verifications                  NaN
neighbourhood                       NaN
calendar_updated                    NaN
instant_bookable                    NaN
Name: price, Length: 66, dtype: float64


### Reflection:
1. Which variables had the strongest positive or negative correlation with price?
2. Which variables might be useful predictors?

### ✍️ Your Response: 🔧
1. The strongest positive correlations with price were price_quote_price_per_night and price_scaled, both at 1.00. Price_per_guest and log_price also had fairly strong positive correlations. I did not notice a strong negative correlation in the results.

2. Some of these variables look strongly related to price, but variables such as price_scaled, log_price, and price_per_guest were created using price, so I would not use them to predict price. Other property features that are not calculated from price would be more useful predictors.

## 4. Define Features and Target Variable

**Business framing:**  
To build a regression model, you need to define what you’re predicting (target) and what you’re using to make that prediction (features).

### Do the following:
- Set `price` as your target variable
- Remove `price` from your predictors

In [13]:
# Keep rows that have a price
model_df = df.dropna(subset=['price']).copy()

# Set price as the target
y = model_df['price']

# Use only numeric columns as predictors
X = model_df.select_dtypes(include='number').copy()

# Remove price and variables created from price
price_cols = [
    'price',
    'price_quote_price_per_night',
    'price_scaled',
    'log_price',
    'price_per_guest'
]

X = X.drop(columns=price_cols, errors='ignore')

# Fill remaining missing values in predictors
X = X.fillna(0)

# Check the features and target
print("X shape:", X.shape)
print("y shape:", y.shape)
print("Missing price values:", y.isna().sum())

X shape: (6943, 57)
y shape: (6943,)
Missing price values: 0


### Reflection:
1. What features are you using?
2. Why is this a regression problem and not a classification problem?

### ✍️ Your Response: 🔧
1. I am using the numeric Airbnb listing features, such as minimum nights, reviews, availability, and property characteristics. I excluded price and variables that were created directly from price.

2. This is a regression problem because price is a numeric value that can have many different values. We are predicting an amount instead of a category or class.

## 5. Split Data into Training and Testing Sets

### Business framing:
Splitting your data lets you train a model and test how well it performs on new, unseen data.

### Do the following:
- Use `train_test_split()` to split into 80% training, 20% testing



In [14]:
from sklearn.model_selection import train_test_split

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

# Check the shapes
print("Training:", X_train.shape)
print("Testing:", X_test.shape)

Training: (5554, 57)
Testing: (1389, 57)


## 6. Fit a Linear Regression Model

### Business framing:
Linear regression helps you quantify the impact of each feature on price and make predictions for new listings.

### Do the following:
- Fit a linear regression model to your training data
- Use it to predict prices for the test set



In [15]:
from sklearn.linear_model import LinearRegression

# Create and train the model
linear_model = LinearRegression()
linear_model.fit(X_train, y_train)

# Predict prices for the test set
y_pred = linear_model.predict(X_test)

# Show first few predictions
print(y_pred[:5])

[349.74904635 349.3298085  349.74552559 349.30972201 357.70638097]


## 7. Evaluate Model Performance

### Business framing:  
A good model should make accurate predictions. We’ll use Mean Squared Error (MSE) and R² to evaluate how close our predictions were to the actual prices.

### Do the following:
- Print MSE and R² score for your model

In [16]:
from sklearn.metrics import mean_squared_error, r2_score

# Calculate model performance
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

# Show results
print("Mean Squared Error (MSE):", mse)
print("R-squared (R2):", r2)

Mean Squared Error (MSE): 107768.06110927706
R-squared (R2): 0.0006612764774286406


### Reflection:
1. What is your R² score? How well does your model explain price variation?
2. Is your MSE large or small? What could you do to improve it?

### ✍️ Your Response: 🔧
1. My R-squared is about 0.001, which is very low. This means the model explains almost none of the variation in Airbnb prices, so the current model does not fit the data very well.

2. The MSE is about 107,768, which seems large. I could try improving the model by using other useful features such as room type or location, handling outliers, or trying a different model.

## 8. Interpret Model Coefficients

### Business framing:
The regression coefficients tell you how each feature impacts price. This can help Airbnb guide hosts and partners.

### Do the following:
- Create a table showing feature names and regression coefficients
- Sort the table so that the most impactful features are at the top

In [17]:
# Create a table of features and coefficients
coef_table = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': linear_model.coef_
})

# Sort by the size of the coefficient
coef_table['Absolute_Coefficient'] = coef_table['Coefficient'].abs()
coef_table = coef_table.sort_values(
    'Absolute_Coefficient',
    ascending=False
)

# Show the most important features
display(coef_table.head(10))

,Feature,Coefficient,Absolute_Coefficient
26,minimum_maximum_nights,2.400371e-08,2.400371e-08
29,maximum_nights_avg_ntm,2.400371e-08,2.400371e-08
27,maximum_maximum_nights,2.400371e-08,2.400371e-08
23,maximum_nights,2.400371e-08,2.400371e-08
41,estimated_revenue_l365d,-3.456117e-13,3.456117e-13
21,price_quote_total_price,-6.152014e-15,6.152014e-15
11,host_listings_count,4.371819e-15,4.371819e-15
40,estimated_occupancy_l365d,-1.083967e-15,1.083967e-15
35,number_of_reviews,-9.025111e-16,9.025111e-16
34,availability_365,-6.481791e-16,6.481791e-16


### Reflection:
1. Which features increased price the most?
2. Were any surprisingly negative?
3. What business insight could you draw from this?

### ✍️ Your Response: 🔧
1. The features related to maximum and minimum nights had the largest positive coefficients, but the coefficient values were still extremely small.

2. Some features, such as estimated_revenue_l365d and price_quote_total_price, had negative coefficients. However, these values were also very close to zero.

3. I think the main business insight is that the numeric features in this model do not explain Airbnb price very well. Other factors, such as location or room type, may be more useful for understanding price.


## 9. Try to Improve the Linear Regression Model

### Business framing:
The first version of your model included all available features — but not all features are equally useful. Removing weak or noisy predictors can often improve performance and interpretation.

### Do the following:
1. Choose your top 3–5 features with the strongest absolute coefficients
2. Rebuild the regression model using just those features
3. Compare MSE and R² between the baseline and refined model

In [18]:
# Choose the top 5 features
top_features = coef_table.head(5)['Feature'].tolist()

print("Top 5 features:")
print(top_features)

# Create training and testing sets with only these features
X_train_refined = X_train[top_features]
X_test_refined = X_test[top_features]

# Create and train the refined model
refined_model = LinearRegression()
refined_model.fit(X_train_refined, y_train)

# Make predictions
y_pred_refined = refined_model.predict(X_test_refined)

# Evaluate the refined model
mse_refined = mean_squared_error(y_test, y_pred_refined)
r2_refined = r2_score(y_test, y_pred_refined)

# Compare the two models
print("\nOriginal Model:")
print("MSE:", mse)
print("R-squared:", r2)

print("\nRefined Model:")
print("MSE:", mse_refined)
print("R-squared:", r2_refined)

Top 5 features:
['minimum_maximum_nights', 'maximum_nights_avg_ntm', 'maximum_maximum_nights', 'maximum_nights', 'estimated_revenue_l365d']

Original Model:
MSE: 107768.06110927706
R-squared: 0.0006612764774286406

Refined Model:
MSE: 100109.412987414
R-squared: 0.07168031086694415


### Reflection:
1. What features did you keep in the refined model, and why?
2. Did model performance improve? Why or why not?
3. Which model would you recommend to stakeholders?
4. How does this relate to your customized learning outcome you created in canvas?

### ✍️ Your Response: 🔧
1. I kept the five features with the largest absolute coefficients: minimum_maximum_nights, maximum_nights_avg_ntm, maximum_maximum_nights, maximum_nights, and estimated_revenue_l365d. I chose them because they had the strongest coefficients in the original model.

2. Yes, the refined model performed better. Its MSE decreased from about 107,768 to 100,109, and its R-squared increased from about 0.001 to 0.072. However, the R-squared is still fairly low.

3. I would recommend the refined model because it performed better and uses fewer features, which also makes it easier to understand. I would still be careful using it for actual pricing decisions because the R-squared is low.

4. This relates to my customized learning outcome because I practiced using Python to analyze data, build a model, compare results, and connect the results to a business decision.


## 10. Reflect and Recommend

### Business framing:  
Ultimately, the value of your model comes from how well it can guide business decisions. Use your results to make real-world recommendations.

### Reflection:
1. What business question did your model help answer?
2. What would you recommend to Airbnb or its hosts?
3. What could you do next to improve this model or make it more useful?
4. How does this relate to your customized learning outcome you created in canvas?


### ✍️ Your Response: 🔧
1. My model helped answer which Airbnb listing features may be related to price and whether those features could be used to predict listing prices.

2. I would recommend that Airbnb or its hosts use the results only as a starting point for understanding pricing. Since the model still has a low R-squared, I would not rely on it alone to set prices.

3. Next, I would include other useful features such as location and room type, handle price outliers, and test other models to see if the predictions improve.

4. This relates to my customized learning outcome because I used Python to analyze real data, build and evaluate a regression model, and use the results to make a business recommendation.

## Submission Instructions
✅ Checklist:
- All code cells run without error
- All markdown responses are complete
- Submit on Canvas as instructed

In [ ]:
!jupyter nbconvert --to html "assignment_11_regression.ipynb"